# E09 OU 过程（数值 ↔ 理论闭环）

> 目标：把 M9 Part 4 的三步推导在 notebook 里逐一对齐：
> 1) 均值回归；2) 方差收敛；3) 自相关指数衰减。

OU SDE：
\[
dX_t = \theta(\mu - X_t)dt + \sigma dW_t,
\quad \theta>0
\]

理论结果：
- 均值：\(m_t = \mu + (m_0-\mu)e^{-\theta t}\)
- 方差（固定初值 \(X_0=x_0\)）：\(v_t=\frac{\sigma^2}{2\theta}(1-e^{-2\theta t})\)
- 稳态自相关：\(\mathrm{Corr}(\tau)=e^{-\theta\tau}\)


In [ ]:
import os
import sys

# Add statphys_urban_learning to sys.path for local imports
curr = os.path.abspath('')
while curr != os.path.dirname(curr):
    if 'statphys_urban_learning' in os.listdir(curr):
        target = os.path.join(curr, 'statphys_urban_learning')
        if target not in sys.path:
            sys.path.insert(0, target)
        break
    curr = os.path.dirname(curr)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from exercises.src.ou import simulate_ou, stationary_variance
from exercises.src.mcmc_diagnostics import integrated_autocorr_time, autocorrelation


## 0) 参数设定（全局共用）


In [ ]:
theta = 2.0
mu = 1.5
sigma = 0.8
x0 = 0.0

# time grid
T = 12.0
dt = 1e-3
n_steps = int(T / dt) + 1

theory_v_inf = stationary_variance(theta, sigma)
print(f"theta={theta}, mu={mu}, sigma={sigma}, x0={x0}, dt={dt}, T={T}")
print(f"theory stationary var = {theory_v_inf:.6f}")


## 1) 均值回归：\(m_t = \mu + (x_0-\mu)e^{-\theta t}\)

这里用多条独立样本路径估计经验均值，再与理论曲线逐点对照。


In [ ]:
n_paths = 256
paths = np.empty((n_paths, n_steps), dtype=float)
for k in range(n_paths):
    tr = simulate_ou(theta=theta, mu=mu, sigma=sigma, x0=x0, dt=dt, n_steps=n_steps, seed=k)
    paths[k] = tr.x

t = tr.t
emp_mean = paths.mean(axis=0)
theory_mean = mu + (x0 - mu) * np.exp(-theta * t)

plt.figure(figsize=(7.2, 4.0))
plt.plot(t, theory_mean, "k--", lw=2, label="theory mean")
plt.plot(t, emp_mean, color="#2563eb", lw=1.8, label=f"empirical mean ({n_paths} paths)")
plt.xlabel("t")
plt.ylabel("mean")
plt.title("OU mean relaxation")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

print("terminal empirical mean:", float(emp_mean[-1]))
print("terminal theory mean   :", float(theory_mean[-1]))


## 2) 方差收敛：\(v_t=\frac{\sigma^2}{2\theta}(1-e^{-2\theta t})\)

在固定初值 \(X_0=x_0\) 下，初始方差为 0，因此方差随时间从 0 指数逼近稳态值。


In [ ]:
emp_var = paths.var(axis=0, ddof=0)
theory_var = theory_v_inf * (1 - np.exp(-2 * theta * t))

plt.figure(figsize=(7.2, 4.0))
plt.plot(t, theory_var, "k--", lw=2, label="theory var")
plt.plot(t, emp_var, color="#f97316", lw=1.8, label=f"empirical var ({n_paths} paths)")
plt.xlabel("t")
plt.ylabel("variance")
plt.title("OU variance relaxation")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

print("terminal empirical var:", float(emp_var[-1]))
print("theory stationary var :", float(theory_v_inf))


## 3) 自相关衰减：\(\mathrm{Corr}(\tau)=e^{-\theta\tau}\)

这里用单条长轨迹（去 burn-in）估计自相关函数，并与理论指数衰减比较。


In [ ]:
T_long = 40.0
n_steps_long = int(T_long / dt) + 1
tr_long = simulate_ou(theta=theta, mu=mu, sigma=sigma, x0=x0, dt=dt, n_steps=n_steps_long, seed=2026)

burn = int(0.25 * len(tr_long.x))
xs = tr_long.x[burn:]

max_lag = 2500
rho = autocorrelation(xs, max_lag=max_lag)
lags = np.arange(len(rho))
tau = lags * dt
rho_theory = np.exp(-theta * tau)

plt.figure(figsize=(7.2, 4.0))
plt.plot(tau, rho, color="#10b981", lw=1.6, label="empirical ACF")
plt.plot(tau, rho_theory, "k--", lw=2.0, label="theory exp(-theta*tau)")
plt.xlim(0, 2.0)
plt.xlabel("tau")
plt.ylabel("Corr(tau)")
plt.title("OU autocorrelation")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

# minimum diagnostics: integrated autocorrelation time and ESS
max_lag_diag = min(5000, len(xs) // 2)
diag = integrated_autocorr_time(xs, max_lag=max_lag_diag)
q = np.exp(-theta * dt)
tau_int_theory = (1 + q) / (1 - q)  # for geometric ACF in discrete sampling
ess_theory = len(xs) / tau_int_theory

print(diag)
print("theory tau_int (discrete):", float(tau_int_theory))
print("theory ESS (approx):", float(ess_theory))


## 讨论（写在你的记录里）

1. 哪一段（均值/方差/自相关）最先与理论对齐？哪一段最慢？为什么？
2. 把 \(\theta\) 改大一倍后：
   - 均值回归速度如何变化？
   - 稳态方差如何变化？
   - 自相关衰减速度如何变化？
3. 把 \(dt\) 放大到 \(5\times 10^{-3}\) 或 \(10^{-2}\) 后，哪类偏差最明显（均值、方差、还是自相关）？
